# Databricks Billing Usage with System Tables

Use Databricks SQL to examine account usage and estimate list-price cost from the `system.billing` schema. Run each cell in order and discuss the result before moving to the next query.

## Learning objectives

- Inspect billing system tables and their schemas.
- Distinguish usage quantity, usage unit, SKU, usage type, and billing product.
- Analyze usage by date, workspace, identity, job, warehouse, and tags.
- Join usage to historical list prices.
- Account for corrections and identify unusual growth.

> Attach the notebook to Databricks compute that can run SQL. Billing data is account-level data. Access requires Unity Catalog and permission to read the billing system tables. An account administrator can grant access when required.

## 1. Confirm access

List the tables available in the billing schema. The two tables used here are:

- `system.billing.usage`: billable usage records;
- `system.billing.list_prices`: historical list prices for Databricks SKUs.

In [ ]:
%sql
SHOW TABLES IN system.billing;

If this query returns a permission error, request `USE CATALOG` on `system`, `USE SCHEMA` on `system.billing`, and `SELECT` on the required tables. Do not continue until access is available.

## 2. Inspect the usage schema

Locate these columns in the result: `usage_date`, `sku_name`, `usage_quantity`, `usage_unit`, `usage_type`, `billing_origin_product`, `usage_metadata`, `identity_metadata`, and `custom_tags`.

In [ ]:
%sql
DESCRIBE TABLE system.billing.usage;

## 3. Read recent billing records

Start with a small sample. Billing timestamps are recorded in UTC. Struct and map columns contain attribution details that vary by workload type.

In [ ]:
%sql
SELECT
  usage_date,
  workspace_id,
  cloud,
  sku_name,
  usage_type,
  usage_unit,
  usage_quantity,
  billing_origin_product,
  record_type,
  identity_metadata,
  usage_metadata,
  custom_tags
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 7 DAYS
ORDER BY usage_start_time DESC
LIMIT 50;

### Observation

`usage_quantity` must always be interpreted with `usage_unit`. Not every record is measured in DBUs. Depending on enabled products, units can represent DBUs, storage, tokens, network usage, or other billable measures.

## 4. Discover usage types and units

Use this query before building a report. It shows which billing measurements actually exist in the account.

In [ ]:
%sql
SELECT
  usage_type,
  usage_unit,
  count(*) AS billing_records,
  round(sum(usage_quantity), 4) AS total_usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY usage_type, usage_unit
ORDER BY total_usage DESC;

## 5. Daily DBU trend

Filter to `usage_unit = 'DBU'` before adding quantities. This prevents unlike units from being combined. Change the visualization to a line chart in Databricks.

In [ ]:
%sql
SELECT
  usage_date,
  round(sum(usage_quantity), 2) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND usage_unit = 'DBU'
GROUP BY usage_date
ORDER BY usage_date;

## 6. Usage by billing product

`billing_origin_product` identifies the Databricks product that originated the usage. A product and a SKU are related but are not the same concept.

In [ ]:
%sql
SELECT
  coalesce(billing_origin_product, 'UNATTRIBUTED') AS product,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= date_trunc('MONTH', current_date())
GROUP BY coalesce(billing_origin_product, 'UNATTRIBUTED'), usage_unit
ORDER BY usage DESC;

## 7. Usage by SKU

A SKU is the billable offering. This breakdown is more detailed than the product view.

In [ ]:
%sql
SELECT
  sku_name,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY sku_name, usage_unit
ORDER BY usage DESC;

## 8. Usage by workspace

Billing system tables contain account-wide records. Grouping by workspace provides the first level of organizational attribution.

In [ ]:
%sql
SELECT
  workspace_id,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY workspace_id, usage_unit
ORDER BY usage DESC;

## 9. Usage by identity

`identity_metadata.run_as` records the user or service principal whose credentials ran the workload. Null values are grouped as unattributed.

In [ ]:
%sql
SELECT
  coalesce(identity_metadata.run_as, 'UNATTRIBUTED') AS run_as,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY coalesce(identity_metadata.run_as, 'UNATTRIBUTED'), usage_unit
ORDER BY usage DESC
LIMIT 25;

## 10. Identify resource attribution fields

Different workload types populate different fields. Review the counts before assuming that every record has a job, cluster, notebook, or warehouse ID.

In [ ]:
%sql
SELECT
  count_if(usage_metadata.job_id IS NOT NULL) AS records_with_job_id,
  count_if(usage_metadata.cluster_id IS NOT NULL) AS records_with_cluster_id,
  count_if(usage_metadata.warehouse_id IS NOT NULL) AS records_with_warehouse_id,
  count_if(usage_metadata.notebook_id IS NOT NULL) AS records_with_notebook_id,
  count(*) AS total_records
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS;

## 11. Most-used jobs

Job attribution is available for serverless jobs and jobs running on job compute. Jobs using all-purpose compute are billed to that compute and may not have a job ID in billing records.

In [ ]:
%sql
SELECT
  workspace_id,
  usage_metadata.job_id AS job_id,
  max(usage_metadata.job_name) AS job_name,
  max(identity_metadata.run_as) AS run_as,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND usage_metadata.job_id IS NOT NULL
GROUP BY workspace_id, usage_metadata.job_id, usage_unit
ORDER BY usage DESC
LIMIT 20;

## 12. SQL warehouse usage

Warehouse IDs connect billing records to SQL compute. This query compares warehouse consumption.

In [ ]:
%sql
SELECT
  workspace_id,
  usage_metadata.warehouse_id AS warehouse_id,
  max(identity_metadata.owned_by) AS warehouse_owner,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND usage_metadata.warehouse_id IS NOT NULL
GROUP BY workspace_id, usage_metadata.warehouse_id, usage_unit
ORDER BY usage DESC;

## 13. Notebook and serverless attribution

Serverless usage can include a notebook path and the identity that ran it. Fields not relevant to a record remain null.

In [ ]:
%sql
SELECT
  coalesce(usage_metadata.notebook_path, 'PATH NOT RECORDED') AS notebook_path,
  coalesce(identity_metadata.run_as, 'UNATTRIBUTED') AS run_as,
  sku_name,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND usage_metadata.notebook_id IS NOT NULL
GROUP BY
  coalesce(usage_metadata.notebook_path, 'PATH NOT RECORDED'),
  coalesce(identity_metadata.run_as, 'UNATTRIBUTED'),
  sku_name,
  usage_unit
ORDER BY usage DESC
LIMIT 25;

## 14. Discover custom tag keys

Custom tags support cost allocation by team, project, environment, or cost center. First discover the keys currently present.

In [ ]:
%sql
SELECT
  tag_key,
  count(*) AS billing_records
FROM system.billing.usage
LATERAL VIEW explode(map_keys(custom_tags)) tags AS tag_key
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY tag_key
ORDER BY billing_records DESC;

## 15. Attribute usage to a tag

Replace `'project'` with a key returned by the preceding query. Missing tags are displayed as `UNTAGGED`.

In [ ]:
%sql
SELECT
  coalesce(custom_tags['project'], 'UNTAGGED') AS project,
  usage_unit,
  round(sum(usage_quantity), 2) AS usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY coalesce(custom_tags['project'], 'UNTAGGED'), usage_unit
ORDER BY usage DESC;

## 16. Inspect historical list prices

Prices are time-bounded. The current price has a null `price_end_time`. The `pricing` struct can contain default and promotional/effective prices.

In [ ]:
%sql
SELECT
  sku_name,
  cloud,
  currency_code,
  usage_unit,
  price_start_time,
  price_end_time,
  pricing
FROM system.billing.list_prices
WHERE price_end_time IS NULL
ORDER BY sku_name
LIMIT 100;

## 17. Estimate list-price cost

Join on SKU and cloud, then match each usage record to the price effective at the usage time. Summing signed usage quantities naturally incorporates retractions and restatements.

> This is an estimated list-price cost, not an invoice. Contract discounts, credits, taxes, commitments, allowances, and cloud-provider infrastructure charges can make invoiced cost different.

In [ ]:
%sql
SELECT
  u.usage_date,
  p.currency_code,
  round(sum(u.usage_quantity * p.pricing.effective_list.default), 2)
    AS estimated_list_cost
FROM system.billing.usage AS u
INNER JOIN system.billing.list_prices AS p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND u.usage_start_time >= p.price_start_time
 AND (u.usage_end_time < p.price_end_time OR p.price_end_time IS NULL)
WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY u.usage_date, p.currency_code
ORDER BY u.usage_date, p.currency_code;

## 18. Cost by product and SKU

This view identifies the products and SKUs contributing most to estimated list cost.

In [ ]:
%sql
SELECT
  coalesce(u.billing_origin_product, 'UNATTRIBUTED') AS product,
  u.sku_name,
  p.currency_code,
  round(sum(u.usage_quantity), 2) AS usage_quantity,
  round(sum(u.usage_quantity * p.pricing.effective_list.default), 2)
    AS estimated_list_cost
FROM system.billing.usage AS u
INNER JOIN system.billing.list_prices AS p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND u.usage_start_time >= p.price_start_time
 AND (u.usage_end_time < p.price_end_time OR p.price_end_time IS NULL)
WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY
  coalesce(u.billing_origin_product, 'UNATTRIBUTED'),
  u.sku_name,
  p.currency_code
ORDER BY estimated_list_cost DESC;

## 19. Cost by workspace and identity

Use this result as a starting point for showback or chargeback. Keep currencies separate.

In [ ]:
%sql
SELECT
  u.workspace_id,
  coalesce(u.identity_metadata.run_as, 'UNATTRIBUTED') AS run_as,
  p.currency_code,
  round(sum(u.usage_quantity * p.pricing.effective_list.default), 2)
    AS estimated_list_cost
FROM system.billing.usage AS u
INNER JOIN system.billing.list_prices AS p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND u.usage_start_time >= p.price_start_time
 AND (u.usage_end_time < p.price_end_time OR p.price_end_time IS NULL)
WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY
  u.workspace_id,
  coalesce(u.identity_metadata.run_as, 'UNATTRIBUTED'),
  p.currency_code
ORDER BY estimated_list_cost DESC
LIMIT 50;

## 20. Understand billing corrections

Billing records can be `ORIGINAL`, `RETRACTION`, or `RESTATEMENT`. Do not remove negative correction records. Aggregate all record types to obtain corrected usage.

In [ ]:
%sql
SELECT
  record_type,
  usage_unit,
  count(*) AS billing_records,
  round(sum(usage_quantity), 4) AS net_usage
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 90 DAYS
GROUP BY record_type, usage_unit
ORDER BY record_type, usage_unit;

## 21. Compare the last two complete months

Exclude the current partial month. `nullif` prevents division by zero when a product had no usage in the earlier month.

In [ ]:
%sql
WITH monthly_usage AS (
  SELECT
    date_trunc('MONTH', usage_date) AS usage_month,
    coalesce(billing_origin_product, 'UNATTRIBUTED') AS product,
    sum(usage_quantity) AS dbus
  FROM system.billing.usage
  WHERE usage_unit = 'DBU'
    AND usage_date >= add_months(date_trunc('MONTH', current_date()), -2)
    AND usage_date < date_trunc('MONTH', current_date())
  GROUP BY date_trunc('MONTH', usage_date),
           coalesce(billing_origin_product, 'UNATTRIBUTED')
),
comparison AS (
  SELECT
    product,
    sum(CASE
          WHEN usage_month = add_months(date_trunc('MONTH', current_date()), -2)
          THEN dbus ELSE 0
        END) AS earlier_month_dbus,
    sum(CASE
          WHEN usage_month = add_months(date_trunc('MONTH', current_date()), -1)
          THEN dbus ELSE 0
        END) AS previous_month_dbus
  FROM monthly_usage
  GROUP BY product
)
SELECT
  product,
  round(earlier_month_dbus, 2) AS earlier_month_dbus,
  round(previous_month_dbus, 2) AS previous_month_dbus,
  round(
    100 * (previous_month_dbus - earlier_month_dbus)
      / nullif(earlier_month_dbus, 0),
    2
  ) AS growth_percent
FROM comparison
ORDER BY growth_percent DESC NULLS LAST;

## 22. Detect high-usage days

Compare each day's DBUs with the preceding seven days. A day is flagged when it is at least 50% above the prior average. This is a simple teaching example, not a complete anomaly-detection model.

In [ ]:
%sql
WITH daily AS (
  SELECT
    usage_date,
    sum(usage_quantity) AS dbus
  FROM system.billing.usage
  WHERE usage_unit = 'DBU'
    AND usage_date >= current_date() - INTERVAL 60 DAYS
  GROUP BY usage_date
),
with_baseline AS (
  SELECT
    usage_date,
    dbus,
    avg(dbus) OVER (
      ORDER BY usage_date
      ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_7_day_average
  FROM daily
)
SELECT
  usage_date,
  round(dbus, 2) AS dbus,
  round(prior_7_day_average, 2) AS prior_7_day_average,
  round(100 * (dbus - prior_7_day_average)
    / nullif(prior_7_day_average, 0), 2) AS percent_above_baseline
FROM with_baseline
WHERE dbus > prior_7_day_average * 1.5
ORDER BY usage_date DESC;

## 23. Data freshness check

Billing system tables update throughout the day and are not real-time. This query shows the latest usage and ingestion timestamps currently available.

In [ ]:
%sql
SELECT
  max(usage_end_time) AS latest_usage_end_time_utc,
  max(ingestion_date) AS latest_ingestion_date,
  datediff(current_date(), max(ingestion_date)) AS ingestion_lag_days
FROM system.billing.usage;

## 24. Practical exercises

Complete each task by modifying a query from the notebook.

1. Return DBUs for the current calendar month by day and product.
2. Find the five most expensive SKUs during the previous complete month.
3. Calculate estimated list cost for one workspace.
4. Find usage with no `run_as` identity and calculate its percentage of total DBUs.
5. Replace the `project` tag with a tag key used in the workspace and calculate cost by tag value.
6. Find the most expensive job runs using `usage_metadata.job_run_id`.
7. Compare weekday and weekend DBU consumption.
8. Create a bar chart of estimated cost by product.
9. Create an alert query that returns a row when yesterday's DBUs exceed a chosen limit.
10. Explain why estimated list-price cost can differ from the final invoice.

## 25. Summary

- Use `system.billing.usage` for account-level usage records.
- Keep different usage units separate when aggregating.
- Use product, SKU, workspace, identity, resource metadata, and tags for attribution.
- Join `system.billing.list_prices` with the correct historical time range to estimate list cost.
- Include retractions and restatements when calculating net usage.
- Expect billing data to arrive with a delay.
- Treat list-price calculations as estimates rather than invoices.

References: [Billable usage table](https://docs.databricks.com/aws/en/admin/system-tables/billing), [pricing table](https://docs.databricks.com/aws/en/admin/system-tables/pricing), and [cost-monitoring queries](https://docs.databricks.com/aws/en/admin/usage/system-tables).